# Final Project: House Price Prediction

This project simulates an end-to-end Machine Learning workflow to predict house prices in California based on various features like location, number of rooms, and median income. It follows the 12-step real-world data science pipeline.

## Step 1: Problem Definition
- **Business Problem:** Predict house prices to help real estate agents and buyers make informed decisions.
- **Prediction:** We are predicting `MedHouseVal` (Median House Value) which is a continuous variable (Regression).
- **Benefits:** Optimizing pricing strategies, understanding feature importance (like location and income).

## Step 2: Data Collection / Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('california_housing.csv')
df.head()

## Step 3: Data Cleaning

In [ ]:
# Check for missing values
print('Missing values:', df.isnull().sum().sum())

# Remove duplicates
df = df.drop_duplicates()

# Check data types
df.info()

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['MedHouseVal'], bins=50, kde=True)
plt.title('Distribution of Median House Value')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

## Step 5: Feature Engineering

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

## Step 6: Splitting the Data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Training shape: {X_train.shape}')
print(f'Testing shape: {X_test.shape}')

## Step 7: Model Selection and Training
We'll evaluate Linear Regression and Random Forest.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

## Step 8: Model Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name} - RMSE: {rmse:.4f}, R2: {r2:.4f}')

evaluate_model('Linear Regression', y_test, lr_preds)
evaluate_model('Random Forest', y_test, rf_preds)

## Step 9: Hyperparameter Tuning
We will use GridSearchCV to tune the Random Forest model.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Defining a smaller grid for faster execution
params = {'n_estimators': [50, 100], 'max_depth': [10, 20, None]}
grid = GridSearchCV(RandomForestRegressor(random_state=42), params, cv=3, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

print('Best Parameters:', grid.best_params_)

## Step 10: Final Model Training and Predictions

In [ ]:
final_model = grid.best_estimator_
final_preds = final_model.predict(X_test)
evaluate_model('Tuned Random Forest', y_test, final_preds)

## Step 11: Conclusion and Insights

In [ ]:
importances = final_model.feature_importances_
feature_names = X.columns

feature_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_df = feature_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x='Importance', y='Feature', data=feature_df)
plt.title('Feature Importances')
plt.show()

- **Insights:** `MedInc` (Median Income) is by far the most crucial determinant of house prices in this dataset, followed by location (`Longitude` and `Latitude`).
- **Business Application:** Real estate agents should heavily weight the income demographics of a neighborhood when pricing new listings.